# ДЗ №13. LDA для 20 Newsgroups

## 1. Импорты

In [1]:
# !pip install gensim scikit-learn datasets pandas numpy

import re
import numpy as np
import pandas as pd

from datasets import load_dataset

from gensim import corpora, models
from gensim.utils import simple_preprocess
from gensim.models.coherencemodel import CoherenceModel

## 2. Загрузка датасета 20_newsgroups(4)

In [2]:
dataset_news = load_dataset("SetFit/20_newsgroups")

categories = [
    "comp.sys.ibm.pc.hardware",
    "comp.sys.mac.hardware",
    "comp.graphics",
    "comp.windows.x"
]

train_news = dataset_news["train"].filter(lambda x: x["label_text"] in categories)
test_news = dataset_news["test"].filter(lambda x: x["label_text"] in categories)

texts = list(train_news["text"]) + list(test_news["text"])
labels = list(train_news["label_text"]) + list(test_news["label_text"])

print("Количество документов:", len(texts))
print("Классы:")
for c in categories:
    print(c, labels.count(c))

'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/SetFit/20_newsgroups/resolve/f1b91292074e7cfb69be58b642d583ec262f30ed/20_newsgroups.py
Retrying in 1s [Retry 1/5].
Using the latest cached version of the dataset since SetFit/20_newsgroups couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\Masha\.cache\huggingface\datasets\SetFit___20_newsgroups\default\0.0.0\f1b91292074e7cfb69be58b642d583ec262f30ed (last modified on Wed Apr 15 13:30:38 2026).


Количество документов: 3906
Классы:
comp.sys.ibm.pc.hardware 982
comp.sys.mac.hardware 963
comp.graphics 973
comp.windows.x 988


## 3. Предобработка текста для LDA

In [3]:
def preprocess_for_lda(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    tokens = simple_preprocess(text, deacc=True, min_len=3)
    return tokens

tokenized_docs = [preprocess_for_lda(text) for text in texts]

print("Пример исходного текста:")
print(texts[0][:500])

print("\nПример токенов:")
print(tokenized_docs[0][:50])

Пример исходного текста:
A fair number of brave souls who upgraded their SI clock oscillator have
shared their experiences for this poll. Please send a brief message detailing
your experiences with the procedure. Top speed attained, CPU rated speed,
add on cards and adapters, heat sinks, hour of usage per day, floppy disk
functionality with 800 and 1.4 m floppies are especially requested.

I will be summarizing in the next two days, so please add to the network
knowledge base if you have done the clock upgrade and haven

Пример токенов:
['fair', 'number', 'brave', 'souls', 'who', 'upgraded', 'their', 'clock', 'oscillator', 'have', 'shared', 'their', 'experiences', 'for', 'this', 'poll', 'please', 'send', 'brief', 'message', 'detailing', 'your', 'experiences', 'with', 'the', 'procedure', 'top', 'speed', 'attained', 'cpu', 'rated', 'speed', 'add', 'cards', 'and', 'adapters', 'heat', 'sinks', 'hour', 'usage', 'per', 'day', 'floppy', 'disk', 'functionality', 'with', 'and', 'floppies', 'are

## 4. Dictionary и Bag-of-Words

Для LDA нужен Bag-of-Words:
- `Dictionary` сопоставляет каждому слову уникальный id
- `doc2bow` превращает документ в список пар `(word_id, count)`

Также удалим слишком редкие и слишком частые слова:
- `no_below=5` — слово должно встретиться хотя бы в 5 документах
- `no_above=0.5` — удаляем слова, встречающиеся более чем в 50% документов

In [4]:
dictionary = corpora.Dictionary(tokenized_docs)

dictionary.filter_extremes(
    no_below=5,
    no_above=0.5,
    keep_n=20000
)

bow_corpus = [dictionary.doc2bow(doc) for doc in tokenized_docs]

print("Размер словаря:", len(dictionary))
print("Пример BoW для первого документа:")
print(bow_corpus[0][:10])

Размер словаря: 5782
Пример BoW для первого документа:
[(0, 1), (1, 2), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 2), (8, 1), (9, 1)]


## 5. Базовая LDA-модель

In [5]:
lda_base = models.LdaModel(
    corpus=bow_corpus,
    id2word=dictionary,
    num_topics=4,
    passes=10,
    iterations=100,
    alpha="symmetric",
    eta="auto",
    random_state=42,
    minimum_probability=0.0
)

topics = lda_base.print_topics(num_words=12)
for topic_id, topic_words in topics:
    print(f"Topic {topic_id}: {topic_words}")

Topic 0: 0.017*"this" + 0.014*"you" + 0.010*"are" + 0.010*"not" + 0.009*"with" + 0.009*"file" + 0.009*"window" + 0.008*"have" + 0.007*"can" + 0.007*"your" + 0.007*"use" + 0.006*"from"
Topic 1: 0.018*"with" + 0.016*"have" + 0.013*"you" + 0.013*"this" + 0.010*"but" + 0.010*"can" + 0.009*"not" + 0.009*"are" + 0.008*"drive" + 0.007*"any" + 0.006*"from" + 0.006*"has"
Topic 2: 0.011*"from" + 0.009*"with" + 0.009*"graphics" + 0.009*"dos" + 0.008*"can" + 0.008*"available" + 0.007*"ftp" + 0.007*"are" + 0.007*"you" + 0.007*"edu" + 0.007*"image" + 0.006*"data"
Topic 3: 0.025*"you" + 0.015*"this" + 0.012*"have" + 0.011*"are" + 0.011*"can" + 0.010*"jpeg" + 0.010*"with" + 0.009*"not" + 0.008*"but" + 0.008*"image" + 0.008*"will" + 0.007*"from"


## 6. Функции для анализа LDA

- показать топ-слова каждой темы
- получить распределение тем для документа
- показать документы и их наиболее вероятные темы
- посчитать `coherence`, чтобы сравнивать модели численно

In [6]:
def print_topics_words(lda_model, num_words=12):
    topics = lda_model.print_topics(num_words=num_words)
    for topic_id, topic_words in topics:
        print(f"Topic {topic_id}: {topic_words}")


def get_document_topic_vectors(lda_model, corpus):
    vectors = []
    for doc_bow in corpus:
        document_topics = lda_model.get_document_topics(doc_bow, minimum_probability=0.0)
        vector = [topic_prob for _, topic_prob in document_topics]
        vectors.append(vector)
    return np.array(vectors)


def show_document_topics(lda_model, corpus, original_texts, true_labels, n_docs=5):
    topic_vectors = get_document_topic_vectors(lda_model, corpus)

    for i in range(n_docs):
        dominant_topic = int(np.argmax(topic_vectors[i]))
        dominant_prob = float(np.max(topic_vectors[i]))

        print("=" * 100)
        print(f"Document {i}")
        print("True class:", true_labels[i])
        print("Dominant topic:", dominant_topic)
        print("Dominant topic probability:", round(dominant_prob, 4))
        print("Topic vector:", np.round(topic_vectors[i], 4))
        print("\nText:")
        print(original_texts[i][:800])


def compute_coherence(lda_model, tokenized_docs, dictionary):
    coherence_model = CoherenceModel(
        model=lda_model,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence="c_v"
    )
    return coherence_model.get_coherence()

## 7. Распределение тем для документов

In [7]:
show_document_topics(
    lda_model=lda_base,
    corpus=bow_corpus,
    original_texts=texts,
    true_labels=labels,
    n_docs=5
)

Document 0
True class: comp.sys.mac.hardware
Dominant topic: 1
Dominant topic probability: 0.9011
Topic vector: [0.0904 0.9011 0.0043 0.0042]

Text:
A fair number of brave souls who upgraded their SI clock oscillator have
shared their experiences for this poll. Please send a brief message detailing
your experiences with the procedure. Top speed attained, CPU rated speed,
add on cards and adapters, heat sinks, hour of usage per day, floppy disk
functionality with 800 and 1.4 m floppies are especially requested.

I will be summarizing in the next two days, so please add to the network
knowledge base if you have done the clock upgrade and haven't answered this
poll. Thanks.
Document 1
True class: comp.sys.mac.hardware
Dominant topic: 1
Dominant topic probability: 0.7487
Topic vector: [0.0015 0.7487 0.0015 0.2483]

Text:
well folks, my mac plus finally gave up the ghost this weekend after
starting life as a 512k way back in 1985.  sooo, i'm in the market for a
new machine a bit sooner than

## 8. Эксперименты с параметрами alpha и beta

In [8]:
experiment_configs = [
    {
        "name": "symmetric_auto",
        "num_topics": 4,
        "alpha": "symmetric",
        "eta": "auto",
        "passes": 10
    },
    {
        "name": "alpha_low_eta_low",
        "num_topics": 4,
        "alpha": 0.01,
        "eta": 0.01,
        "passes": 10
    },
    {
        "name": "alpha_low_eta_medium",
        "num_topics": 4,
        "alpha": 0.01,
        "eta": 0.1,
        "passes": 10
    },
    {
        "name": "alpha_medium_eta_low",
        "num_topics": 4,
        "alpha": 0.1,
        "eta": 0.01,
        "passes": 10
    },
    {
        "name": "alpha_medium_eta_medium",
        "num_topics": 4,
        "alpha": 0.1,
        "eta": 0.1,
        "passes": 10
    },
    {
        "name": "alpha_high_eta_high",
        "num_topics": 4,
        "alpha": 1.0,
        "eta": 1.0,
        "passes": 10
    },
    {
        "name": "auto_auto",
        "num_topics": 4,
        "alpha": "auto",
        "eta": "auto",
        "passes": 10
    },
]

## 9. Обучение нескольких LDA-моделей

In [9]:
lda_models = {}
lda_results = []

for config in experiment_configs:
    print("\n" + "=" * 100)
    print("Training:", config["name"])
    print("alpha:", config["alpha"], "eta:", config["eta"])

    lda = models.LdaModel(
        corpus=bow_corpus,
        id2word=dictionary,
        num_topics=config["num_topics"],
        passes=config["passes"],
        iterations=100,
        alpha=config["alpha"],
        eta=config["eta"],
        random_state=42,
        minimum_probability=0.0
    )

    topic_vectors = get_document_topic_vectors(lda, bow_corpus)
    max_topic_probs = topic_vectors.max(axis=1)
    coherence = compute_coherence(lda, tokenized_docs, dictionary)

    lda_models[config["name"]] = lda

    lda_results.append({
        "model": config["name"],
        "num_topics": config["num_topics"],
        "alpha": str(config["alpha"]),
        "eta_beta": str(config["eta"]),
        "coherence_cv": coherence,
        "mean_max_topic_prob": max_topic_probs.mean(),
        "std_max_topic_prob": max_topic_probs.std()
    })

    print("Coherence:", round(coherence, 4))
    print("Mean max topic probability:", round(max_topic_probs.mean(), 4))

lda_results_df = pd.DataFrame(lda_results).sort_values("coherence_cv", ascending=False)
lda_results_df


Training: symmetric_auto
alpha: symmetric eta: auto
Coherence: 0.3841
Mean max topic probability: 0.789

Training: alpha_low_eta_low
alpha: 0.01 eta: 0.01
Coherence: 0.3844
Mean max topic probability: 0.7997

Training: alpha_low_eta_medium
alpha: 0.01 eta: 0.1
Coherence: 0.3844
Mean max topic probability: 0.8094

Training: alpha_medium_eta_low
alpha: 0.1 eta: 0.01
Coherence: 0.3844
Mean max topic probability: 0.7849

Training: alpha_medium_eta_medium
alpha: 0.1 eta: 0.1
Coherence: 0.3844
Mean max topic probability: 0.7936

Training: alpha_high_eta_high
alpha: 1.0 eta: 1.0
Coherence: 0.3747
Mean max topic probability: 0.6244

Training: auto_auto
alpha: auto eta: auto
Coherence: 0.3841
Mean max topic probability: 0.8285


,model,num_topics,alpha,eta_beta,coherence_cv,mean_max_topic_prob,std_max_topic_prob
1,alpha_low_eta_low,4,0.01,0.01,0.384357,0.799698,0.203985
2,alpha_low_eta_medium,4,0.01,0.1,0.384357,0.809371,0.203058
3,alpha_medium_eta_low,4,0.1,0.01,0.384357,0.784869,0.200397
4,alpha_medium_eta_medium,4,0.1,0.1,0.384357,0.793581,0.199905
0,symmetric_auto,4,symmetric,auto,0.384146,0.788963,0.197666
6,auto_auto,4,auto,auto,0.384146,0.828539,0.189435
5,alpha_high_eta_high,4,1.0,1.0,0.374652,0.624414,0.167857


## 10. Топ-слова тем для всех конфигураций

Какие слова формируют темы при разных значениях `alpha` и `eta`.

In [10]:
for model_name, lda in lda_models.items():
    print("\n" + "=" * 120)
    print("MODEL:", model_name)
    print("=" * 120)
    print_topics_words(lda, num_words=12)


MODEL: symmetric_auto
Topic 0: 0.017*"this" + 0.014*"you" + 0.010*"are" + 0.010*"not" + 0.009*"with" + 0.009*"file" + 0.009*"window" + 0.008*"have" + 0.007*"can" + 0.007*"your" + 0.007*"use" + 0.006*"from"
Topic 1: 0.018*"with" + 0.016*"have" + 0.013*"you" + 0.013*"this" + 0.010*"but" + 0.010*"can" + 0.009*"not" + 0.009*"are" + 0.008*"drive" + 0.007*"any" + 0.006*"from" + 0.006*"has"
Topic 2: 0.011*"from" + 0.009*"with" + 0.009*"graphics" + 0.009*"dos" + 0.008*"can" + 0.008*"available" + 0.007*"ftp" + 0.007*"are" + 0.007*"you" + 0.007*"edu" + 0.007*"image" + 0.006*"data"
Topic 3: 0.025*"you" + 0.015*"this" + 0.012*"have" + 0.011*"are" + 0.011*"can" + 0.010*"jpeg" + 0.010*"with" + 0.009*"not" + 0.008*"but" + 0.008*"image" + 0.008*"will" + 0.007*"from"

MODEL: alpha_low_eta_low
Topic 0: 0.018*"this" + 0.014*"you" + 0.010*"are" + 0.010*"not" + 0.009*"with" + 0.009*"file" + 0.008*"window" + 0.008*"have" + 0.008*"your" + 0.007*"can" + 0.007*"use" + 0.006*"from"
Topic 1: 0.018*"with" + 0.01

## 11. Сравнение документов: какие темы формируют документ

In [11]:
best_model_name = lda_results_df.iloc[0]["model"]
best_lda = lda_models[best_model_name]

print("Best model:", best_model_name)

show_document_topics(
    lda_model=best_lda,
    corpus=bow_corpus,
    original_texts=texts,
    true_labels=labels,
    n_docs=8
)

Best model: alpha_low_eta_low
Document 0
True class: comp.sys.mac.hardware
Dominant topic: 1
Dominant topic probability: 0.6953
Topic vector: [9.840e-02 6.953e-01 2.000e-04 2.062e-01]

Text:
A fair number of brave souls who upgraded their SI clock oscillator have
shared their experiences for this poll. Please send a brief message detailing
your experiences with the procedure. Top speed attained, CPU rated speed,
add on cards and adapters, heat sinks, hour of usage per day, floppy disk
functionality with 800 and 1.4 m floppies are especially requested.

I will be summarizing in the next two days, so please add to the network
knowledge base if you have done the clock upgrade and haven't answered this
poll. Thanks.
Document 1
True class: comp.sys.mac.hardware
Dominant topic: 1
Dominant topic probability: 0.5515
Topic vector: [1.000e-04 5.515e-01 1.000e-04 4.484e-01]

Text:
well folks, my mac plus finally gave up the ghost this weekend after
starting life as a 512k way back in 1985.  sooo,

## 12. Распределение тем документов (таблица)

In [12]:
topic_vectors = get_document_topic_vectors(best_lda, bow_corpus)

doc_topic_rows = []

for i in range(10):
    dominant_topic = int(np.argmax(topic_vectors[i]))
    dominant_prob = float(np.max(topic_vectors[i]))

    row = {
        "document_id": i,
        "true_label": labels[i],
        "dominant_topic": dominant_topic,
        "dominant_topic_prob": dominant_prob,
        "text_start": texts[i][:200].replace("\n", " ")
    }

    for topic_id in range(topic_vectors.shape[1]):
        row[f"topic_{topic_id}"] = topic_vectors[i][topic_id]

    doc_topic_rows.append(row)

doc_topic_df = pd.DataFrame(doc_topic_rows)
doc_topic_df

,document_id,true_label,dominant_topic,dominant_topic_prob,text_start,topic_0,topic_1,topic_2,topic_3
0,0,comp.sys.mac.hardware,1,0.695305,A fair number of brave souls who upgraded thei...,0.098359,0.695305,0.000159,0.206176
1,1,comp.sys.mac.hardware,1,0.551420,"well folks, my mac plus finally gave up the gh...",0.000057,0.551420,0.000057,0.448466
2,2,comp.graphics,1,0.997699,Do you have Weitek's address/phone number? I...,0.000767,0.997699,0.000767,0.000767
3,3,comp.sys.ibm.pc.hardware,1,0.963611,...,0.000056,0.963611,0.036278,0.000056
4,4,comp.sys.mac.hardware,1,0.687173,"I've had the board for over a year, and it ...",0.000103,0.687173,0.000103,0.312621
5,5,comp.sys.mac.hardware,0,0.250000,--,0.250000,0.250000,0.250000,0.250000
6,6,comp.graphics,3,0.923770,I certainly do use it whenever I have to do T...,0.054157,0.022009,0.000064,0.923770
7,7,comp.windows.x,0,0.683465,QUESTION: What is the EXACT entry (parameter...,0.683465,0.153549,0.162883,0.000103
8,8,comp.sys.mac.hardware,1,0.714578,I don't know about the specific problem mentio...,0.284895,0.714578,0.000263,0.000263
9,9,comp.graphics,3,0.469218,"Hello, I am looking to add voice input ca...",0.000133,0.158946,0.371703,0.469218


## 13. Дополнительный эксперимент: разное число тем

In [13]:
num_topics_results = []
num_topics_models = {}

for n_topics in [4, 6, 8, 10]:
    print("\nTraining num_topics =", n_topics)

    lda = models.LdaModel(
        corpus=bow_corpus,
        id2word=dictionary,
        num_topics=n_topics,
        passes=10,
        iterations=100,
        alpha="auto",
        eta="auto",
        random_state=42,
        minimum_probability=0.0
    )

    coherence = compute_coherence(lda, tokenized_docs, dictionary)
    topic_vectors = get_document_topic_vectors(lda, bow_corpus)
    max_topic_probs = topic_vectors.max(axis=1)

    num_topics_models[n_topics] = lda

    num_topics_results.append({
        "num_topics": n_topics,
        "alpha": "auto",
        "eta_beta": "auto",
        "coherence_cv": coherence,
        "mean_max_topic_prob": max_topic_probs.mean()
    })

num_topics_df = pd.DataFrame(num_topics_results).sort_values("coherence_cv", ascending=False)
num_topics_df


Training num_topics = 4

Training num_topics = 6

Training num_topics = 8

Training num_topics = 10


,num_topics,alpha,eta_beta,coherence_cv,mean_max_topic_prob
3,10,auto,auto,0.409292,0.634090
2,8,auto,auto,0.407925,0.661826
1,6,auto,auto,0.405429,0.748934
0,4,auto,auto,0.384146,0.828539


## 14. Темы лучшей модели по числу тем

In [14]:
best_n_topics = int(num_topics_df.iloc[0]["num_topics"])
best_num_topics_lda = num_topics_models[best_n_topics]

print("Best num_topics:", best_n_topics)
print_topics_words(best_num_topics_lda, num_words=12)

Best num_topics: 10
Topic 0: 0.019*"this" + 0.015*"with" + 0.014*"you" + 0.014*"have" + 0.013*"was" + 0.011*"but" + 0.010*"problem" + 0.010*"not" + 0.009*"from" + 0.009*"when" + 0.008*"drive" + 0.007*"has"
Topic 1: 0.020*"with" + 0.018*"have" + 0.014*"this" + 0.012*"card" + 0.011*"can" + 0.011*"you" + 0.011*"are" + 0.009*"any" + 0.009*"but" + 0.008*"not" + 0.008*"what" + 0.007*"one"
Topic 2: 0.042*"scsi" + 0.034*"drive" + 0.019*"with" + 0.017*"ide" + 0.017*"controller" + 0.016*"drives" + 0.015*"bus" + 0.013*"disk" + 0.011*"this" + 0.011*"you" + 0.010*"hard" + 0.010*"have"
Topic 3: 0.021*"you" + 0.015*"have" + 0.015*"with" + 0.012*"but" + 0.011*"mhz" + 0.011*"are" + 0.011*"would" + 0.011*"not" + 0.011*"about" + 0.011*"they" + 0.008*"like" + 0.008*"this"
Topic 4: 0.020*"this" + 0.016*"have" + 0.016*"you" + 0.014*"can" + 0.012*"but" + 0.012*"with" + 0.012*"any" + 0.010*"window" + 0.009*"not" + 0.009*"would" + 0.008*"there" + 0.008*"are"
Topic 5: 0.024*"file" + 0.021*"entry" + 0.017*"outpu